# Part-damage classification

Train three ResNet18 models from ImageNet on reviewed real part crops, then fine-tune on synthetic crops. Select checkpoints on real validation data and evaluate on real test data. Completed stages are validated and reused.

## Setup

Install the root `requirements.txt` in your chosen Python environment and select its kernel. See [setup instructions](../../README.md#run). Run cells in order. ImageNet weights download on the first training run if not cached.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import json
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/taxonomy.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the CRATER repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import torch
from tqdm.auto import tqdm
from PIL import Image, ImageDraw
from IPython.display import display
from tools.data.import_cvat_damage_annotations import sha256_file
from tools.data.prepare_damage_classification import (
    REVIEW_FIELDS, load_inventory, proposed_review_rows, write_csv, read_csv,
    apply_review, materialize_crops,
)
from experiments.damage.training import (
    TrainingConfig, load_completed_stage, train_stage, final_comparison,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}; device={DEVICE}")

## Parameters

Use a new `RUN_NAME` when changing data or training settings. Synthetic fine-tuning starts from each real model's best checkpoint at a lower learning rate.

In [ ]:
RUN_NAME = "humvee_damage_real_then_synthetic_v1"
RUN_DIR = ROOT / "outputs/damage/humvee_cvat" / RUN_NAME
CROP_DIR = RUN_DIR / "data"
REVIEW_PATH = ROOT / "datasets/military/damage/review/humvee_real_synthetic_v1.csv"
PART_GROUPS = ["mobility", "structure", "mission_equipment"]
LEVELS = json.loads((ROOT / "configs/taxonomy.json").read_text())["damage"]["levels"]
INITIALIZATION = "imagenet"
REAL_CONFIG = TrainingConfig(epochs=20, learning_rate=1e-4, batch_size=32, seed=42)
SYNTHETIC_CONFIG = TrainingConfig(epochs=10, learning_rate=1e-5, batch_size=32, seed=42)
print("Damage label order:", LEVELS)
print("Run directory:", RUN_DIR)

## 1. Verify source inventories

Check source hashes, dimensions, manual labels, box geometry and taxonomy.

In [ ]:
inventory = load_inventory(ROOT, include_synthetic=True)
inventory_table = pd.DataFrame(inventory)
display(inventory_table.groupby("domain").agg(
    images=("source_image", "nunique"), boxes=("annotation_id", "count")))
display(pd.crosstab([inventory_table.domain, inventory_table.component_group],
                    inventory_table.damage_level).reindex(columns=LEVELS, fill_value=0))

## 2. Review images and groups

Open the CSV below. Check part identity, box fit and damage labels, including default `intact` values. Merge related scene/vehicle photos into the same group, then mark every image `accept` or `reject`. Synthetic review also covers generation artifacts. Pending rows block crop preparation; see [labeling rules](../../docs/labeling.md).

In [ ]:
if not REVIEW_PATH.exists():
    REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
    write_csv(REVIEW_PATH, proposed_review_rows(inventory), REVIEW_FIELDS)
review_table = pd.DataFrame(read_csv(REVIEW_PATH))
print("Edit this review CSV:", REVIEW_PATH)
display(review_table.groupby(["domain", "review_status"]).size().rename("images").to_frame())
display(review_table.head(8))

### Inspect an annotated image

Change `REVIEW_IMAGE_INDEX` to inspect another image. Use the annotation IDs for corrections in CVAT.

In [ ]:
REVIEW_IMAGE_INDEX = 0
review_image = review_table.iloc[REVIEW_IMAGE_INDEX]
image_rows = inventory_table[
    (inventory_table.domain == review_image.domain)
    & (inventory_table.source_image == review_image.source_image)
]
source_path = ROOT / image_rows.iloc[0].image_path
with Image.open(source_path) as source:
    preview = source.convert("RGB")
draw = ImageDraw.Draw(preview)
for number, (_, row) in enumerate(image_rows.iterrows()):
    box = tuple(float(row[f"box_{axis}"]) for axis in ("x1", "y1", "x2", "y2"))
    draw.rectangle(box, outline="lime", width=3)
    draw.text((box[0], box[1]), str(number), fill="yellow", stroke_width=2, stroke_fill="black")
preview.thumbnail((1400, 1000))
print(source_path)
display(preview)
display(image_rows[["annotation_id", "component_class", "damage_state_source", "damage_level"]].reset_index(drop=True))

## 3. Prepare splits and crops

Split reviewed real groups approximately 70/15/15 before cropping. Every crop keeps its source split; synthetic crops enter training only. Check class coverage below.

In [ ]:
reviewed = apply_review(inventory, REVIEW_PATH, seed=REAL_CONFIG.seed)
preparation = {
    "review_sha256": sha256_file(REVIEW_PATH),
    "seed": REAL_CONFIG.seed,
    "taxonomy_sha256": sha256_file(ROOT / "configs/taxonomy.json"),
    "sources": sorted({(r["domain"], r["source_labels_sha256"], r["source_manifest_sha256"])
                       for r in inventory}),
}
# JSON normalization makes saved and in-memory tuples/lists comparable.
preparation = json.loads(json.dumps(preparation))
if CROP_DIR.exists():
    metadata_file = CROP_DIR / "preparation.json"
    if not metadata_file.is_file() or json.loads(metadata_file.read_text()) != preparation:
        raise ValueError("Incomplete or changed prepared data. Choose a new RUN_NAME.")
    crop_rows = read_csv(CROP_DIR / "crops_manifest.csv")
else:
    crop_rows = materialize_crops(ROOT, reviewed, CROP_DIR)
    (CROP_DIR / "review_snapshot.csv").write_bytes(REVIEW_PATH.read_bytes())
    (CROP_DIR / "preparation.json").write_text(json.dumps(preparation, indent=2) + "\n")
crop_table = pd.DataFrame(crop_rows)
coverage = pd.crosstab([crop_table.domain, crop_table.split, crop_table.component_group],
                      crop_table.damage_level).reindex(columns=LEVELS, fill_value=0)
display(coverage)
coverage.to_csv(CROP_DIR / "class_coverage.csv")
print("Prepared crops:", len(crop_rows), "Manifest:", CROP_DIR / "crops_manifest.csv")

## 4. Check training coverage

Each selected group needs two observed training classes in each stage and nonempty real validation/test sets. Missing classes remain explicit in the four-output model.

In [ ]:
problems = []
for group in PART_GROUPS:
    for domain in ("real", "synthetic"):
        selected = crop_table[(crop_table.component_group == group)
                              & (crop_table.domain == domain) & (crop_table.split == "train")]
        if len(selected) < 2 or selected.damage_level.nunique() < 2:
            problems.append(f"{group}/{domain}: need >=2 training crops and >=2 observed damage classes")
    for split in ("val", "test"):
        selected = crop_table[(crop_table.component_group == group)
                              & (crop_table.domain == "real") & (crop_table.split == split)]
        if selected.empty:
            problems.append(f"{group}: no real {split} crops")
if problems:
    raise ValueError("\n".join(problems) + "\nCollect/review more data or explicitly defer that part group in PART_GROUPS.")
print("Selected part groups have data for both training stages and real evaluation.")

## 5. Train on real crops

Train independent ResNet18 models for mobility, structure and mission equipment. Select `best.pt` by real validation macro F1; `last.pt` preserves the final epoch. Completed matching stages are reused. Incomplete or changed runs require inspection and a new run name.

In [ ]:
real_checkpoints = {}
for group in PART_GROUPS:
    arguments = dict(
        rows=crop_rows, crop_root=CROP_DIR, output_dir=RUN_DIR / "real" / group,
        group=group, levels=LEVELS, domain="real", config=REAL_CONFIG,
        initialization=INITIALIZATION,
    )
    checkpoint = load_completed_stage(**arguments)
    if checkpoint is None:
        checkpoint = train_stage(**arguments, device=DEVICE)
    else:
        print(f"{group}/real: reusing validated completed stage at {checkpoint}")
    real_checkpoints[group] = checkpoint

## 6. Fine-tune on synthetic crops

Load each real `best.pt` and continue with synthetic crops only. Epoch 0 records the unchanged real baseline. If validation never improves, synthetic `best.pt` keeps epoch 0; `last.pt` preserves the final adaptation.

In [ ]:
synthetic_checkpoints = {}
for group in PART_GROUPS:
    arguments = dict(
        rows=crop_rows, crop_root=CROP_DIR, output_dir=RUN_DIR / "synthetic" / group,
        group=group, levels=LEVELS, domain="synthetic", config=SYNTHETIC_CONFIG,
        parent_checkpoint=real_checkpoints[group],
    )
    checkpoint = load_completed_stage(**arguments)
    if checkpoint is None:
        checkpoint = train_stage(**arguments, device=DEVICE)
    else:
        print(f"{group}/synthetic: reusing validated completed stage at {checkpoint}")
    synthetic_checkpoints[group] = checkpoint

## 7. Compare validation and select checkpoints

Choose between stages using real validation macro F1. Epoch 0 means synthetic training did not earn a replacement. Save scores, epoch histories and selected-checkpoint hashes. Tables avoid the Matplotlib/OpenMP conflict in this Windows environment.

In [ ]:
comparison_rows, history_rows, selected_checkpoints = [], [], {}
for group in tqdm(PART_GROUPS, desc="Comparing stages"):
    real = torch.load(real_checkpoints[group], map_location="cpu", weights_only=True)
    synthetic = torch.load(synthetic_checkpoints[group], map_location="cpu", weights_only=True)
    delta = synthetic["validation"]["macro_f1"] - real["validation"]["macro_f1"]
    selected_checkpoints[group] = synthetic_checkpoints[group] if delta > 0 else real_checkpoints[group]
    comparison_rows.append({"group": group, "real_macro_f1": real["validation"]["macro_f1"],
                            "synthetic_macro_f1": synthetic["validation"]["macro_f1"],
                            "delta": delta, "synthetic_best_epoch": synthetic["epoch"],
                            "selected_stage": "synthetic" if delta > 0 else "real"})
    for stage in ("real", "synthetic"):
        history = json.loads((RUN_DIR / stage / group / "history.json").read_text())
        history_rows.extend({"group": group, "stage": stage, "epoch": row["epoch"],
                             "train_loss": row["train_loss"],
                             "real_validation_macro_f1": row["validation"]["macro_f1"]}
                            for row in history)
validation_comparison = pd.DataFrame(comparison_rows)
validation_history = pd.DataFrame(history_rows)
validation_comparison.to_csv(RUN_DIR / "validation_comparison.csv", index=False)
validation_history.to_csv(RUN_DIR / "validation_history.csv", index=False)
selection = {group: {"path": str(path.relative_to(RUN_DIR)), "sha256": sha256_file(path)}
             for group, path in selected_checkpoints.items()}
(RUN_DIR / "selected_checkpoints.json").write_text(json.dumps(selection, indent=2) + "\n")
display(validation_comparison)
display(validation_history)

## 8. Evaluate on real test crops

Evaluate after selection is fixed. Save per-crop predictions, confusion matrices and metrics for both stages. Reruns validate and reload saved test results. Do not use test scores to tune settings.

In [ ]:
TEST_DIR = RUN_DIR / "test_final"
paired_checkpoints = {f"{group}_{stage}": paths[group]
                      for stage, paths in (("real", real_checkpoints), ("synthetic", synthetic_checkpoints))
                      for group in PART_GROUPS}
if TEST_DIR.exists():
    result_file = TEST_DIR / "comparison.json"
    if not result_file.is_file():
        raise ValueError("Final test output is incomplete; inspect it before attempting another evaluation.")
    test_results = json.loads(result_file.read_text())
    if set(test_results) != set(paired_checkpoints) or any(
        test_results[name]["checkpoint_sha256"] != sha256_file(path)
        for name, path in paired_checkpoints.items()
    ):
        raise ValueError("Saved final test results belong to different checkpoints.")
else:
    test_results = final_comparison(paired_checkpoints, crop_rows, CROP_DIR, TEST_DIR, DEVICE)
display(pd.DataFrame([{"model": name, **{key: result[key] for key in
    ("samples", "macro_f1", "accuracy", "ece_10_bins", "brier_score")}}
    for name, result in test_results.items()]))

## 9. Inspect class support and confusion

Rows are true labels; columns are predictions. Absent classes are coverage gaps. Use the prediction CSVs to inspect individual component types.

In [ ]:
MODEL_TO_INSPECT = f"{PART_GROUPS[0]}_synthetic"
result = test_results[MODEL_TO_INSPECT]
display(pd.DataFrame(result["per_class"]).T)
confusion = result["confusion_matrix"]
confusion_table = pd.DataFrame(confusion, index=LEVELS, columns=LEVELS)
confusion_table.index.name = "true_label"
confusion_table.columns.name = "predicted_label"
display(confusion_table)

## Results

Outputs are under `RUN_DIR`: `validation_comparison.csv`, `validation_history.csv`, `selected_checkpoints.json` and `test_final/`. These scores evaluate human boxes; combined detector-to-damage performance remains unmeasured.